<div align="center">

####
#### Máster Universitario en Ingeniería Biomédica y Salud Digital
#### Trabajo Fin de Máster
<br>

**Diseño y desarrollo de un modelo de Machine Learning para predicción de la propagación del Dengue**
<br><br>
</div>
<div align="left">
Realizado por: Carlos Lazzarino <br>
Dirigido por: Isabel Nepomuceno Chamorro

</div>


#Instalación de paquetes y carga de datos

In [1]:
# Configuración preliminar
# En Colab se requiere copiar y pegar los archivos en bruto en la carpeta del Colab (en un futuro se incorporará carga directa desde un drive)
# Asegurar entorno de ejecución en R
# En ocasiones puede ser necesario cambiar en tipo de entorno de ejecución en "Acelerador de hardware" a GPU (T4 o similar).

Instalar y cargar librerias

In [2]:
# Esta parte revisa si ya están cargadas las librerias, en ocasiones algunas ya están instaladas en Colab

packages <- c("data.table", "lightgbm", "zoo", "rBayesianOptimization", "e1071")
packages_faltantes <- packages[!(packages %in% installed.packages()[,"Package"])]
if(length(packages_faltantes) > 0) {
  cat("Instalando paquetes faltantes:", paste(packages_faltantes, collapse=", "), "\n")
  install.packages(packages_faltantes)
} else {
  cat("Todos los paquetes ya están instalados.\n")
}
invisible(lapply(packages, library, character.only = TRUE))
cat("Todas las librerías están cargadas")


Todos los paquetes ya están instalados.



Attaching package: ‘data.table’


The following object is masked from ‘package:base’:

    %notin%



Attaching package: ‘zoo’


The following objects are masked from ‘package:data.table’:

    yearmon, yearqtr


The following objects are masked from ‘package:base’:

    as.Date, as.Date.numeric




Todas las librerías están cargadas

In [3]:
#Keras/Tensorflow falla ocasionalmente en algunos entornos R, dependiendo la versión, por lo que lo hice por separado y utilicé una librería nativa de R (Torch)
#Para keras, se utiliza reticulate que es una interfaz puente que permite ejecutar código Python dentro de una sesión de R y se instala keras3,
#Las versiones anteriores son obsoletas y podrían no correr en Colab. Esta instalación puede demorar unos 15-20 minutos.
# Se instala entonces Torch, creado por POSIT:

if (!("torch" %in% installed.packages()[,"Package"])) {
  cat("Instalando el paquete base 'torch'...\n")
  install.packages("torch")
}
library(torch)

if (!torch_is_installed()) {
  cat("Descargando el motor C++ libtorch (esto se hace una sola vez y pesa ~1.5GB)...\n")
  install_torch()
}

if (!("luz" %in% installed.packages()[,"Package"])) {
  cat("Instalando 'luz' (es un wrapper estilo Keras para Torch)...\n")
  install.packages("luz")
}
library(luz)

#Dejo comentado el código para keras3
#if (!("reticulate" %in% installed.packages()[,"Package"])) install.packages("reticulate")
#library(reticulate)
#cat("Package Reticulate cargado. \n")
#if (!("keras3" %in% installed.packages()[,"Package"])) install.packages("keras3")
#library(keras3)
#cat("Package Keras3 cargado. \n")
#reticulate::install_python(version = "3.11.15")
#cat("Entorno Python instalado. \n")
#keras3::install_keras()
#cat("Keras3 instalado. \n")

Carga de datos

In [4]:
# en colab se requiere copiar y pegar los archivos en bruto en la carpeta del Colab

train_features <- fread("dengue_features_train.csv")
train_labels <- fread("dengue_labels_train.csv")
test_features <- fread("dengue_features_test.csv")
submission_format <- fread("submission_format.csv")

Armado del dataset y preprocesamiento

In [5]:
# Unión de las features y labels de train que vienen separadas
train_data <- merge(train_features, train_labels, by=c("city", "year", "weekofyear"))

# Se añade una columna vacía a test para poder concatenar todo
test_features[, total_cases := NA]
dataset <- rbindlist(list(train_data, test_features), use.names=TRUE, fill=TRUE)

cat("Datos combinados. Filas totales:", nrow(dataset), "\n")

# 3. Preprocesamiento (Imputación de los NA por LOCF Last Observation Carried Forward) -hay NA en el índice de vegetación en el database original-
setorder(dataset, city, year, weekofyear)

# Definimos las columnas numéricas ambientales
cols_climaticas <- setdiff(names(dataset), c("city", "year", "weekofyear", "week_start_date", "total_cases"))

# Imputamos arrastrando el último valor válido (LOCF) y luego el próximo (NOCB) por si hay nulos al inicio
dataset[, (cols_climaticas) := lapply(.SD, function(x) na.locf(x, na.rm=FALSE)), by=city, .SDcols=cols_climaticas]
dataset[, (cols_climaticas) := lapply(.SD, function(x) na.locf(x, fromLast=TRUE, na.rm=FALSE)), by=city, .SDcols=cols_climaticas]


Datos combinados. Filas totales: 1872 


# Feature Engineering

In [6]:

# Crea una nueva columna llamada temp_rango (que corresponde a la amplitud térmica) y una que calcula el promedio para esa semana del Indice de Vegetación (ndvi) a partir de los 4 valores aportados en el dataset.

dataset[, temp_rango := station_max_temp_c - station_min_temp_c]
dataset[, ndvi_promedio := rowMeans(cbind(ndvi_ne, ndvi_nw, ndvi_se, ndvi_sw), na.rm=TRUE)]

In [7]:
# Feature Engineering (Lags y Deltas)

# Crea una copia del valor de esa variable pero desplazada 1 semana, 2, 3 y 4 semanas hacia atrás en el tiempo... tiempo del vector biológico
cat("Calculando Lags temporales (4 semanas retrospectivas)...\n")
lags <- c(1, 2, 3, 4)
cols_a_laguear <- c(cols_climaticas, "temp_rango", "ndvi_promedio")

for(l in lags) {
  dataset[, paste0(cols_a_laguear, "_lag", l) := shift(.SD, l, type="lag"), by=city, .SDcols=cols_a_laguear]
}

# Calcula la diferencia entre el valor y lag1
cat("Calculando Deltas (Variación semanal)...\n")
for(col in cols_a_laguear) {
  dataset[, paste0(col, "_delta1") := get(col) - get(paste0(col, "_lag1"))]
}

Calculando Lags temporales (4 semanas retrospectivas)...
Calculando Deltas (Variación semanal)...


In [8]:
# Promedios Móviles (Tendencias de 4 y 8 semanas), son los promedios en ventana movil (para estudiar efecto acumulativo)

cat("Calculando Promedios Móviles estacionales...\n")
for(col in cols_a_laguear) {
  dataset[, paste0(col, "_rmean4") := frollmean(get(col), 4, align="right", na.rm=TRUE), by=city]
  dataset[, paste0(col, "_rmean8") := frollmean(get(col), 8, align="right", na.rm=TRUE), by=city]
}

Calculando Promedios Móviles estacionales...


In [9]:
# Se vuelve a aplicar LOCF y NOCB en todas las columnas que tengan NA
# porque los promedios moviles, los lags y los delta generarán NA cuando el periodo de cálculo exceda el disponible en la base
# Si hay NA va a fallar SVM

cols_todas <- setdiff(names(dataset), c("city", "year", "weekofyear", "week_start_date", "total_cases"))
dataset[, (cols_todas) := lapply(.SD, function(x) na.locf(na.locf(x, na.rm=FALSE), fromLast=TRUE, na.rm=FALSE)), by=city, .SDcols=cols_todas]

# Eliminar week_start_date porque no es numérica y es redundante con la semana y el año que ya está presente
dataset[, week_start_date := NULL]


In [10]:
# Se separan los datos por ciudad (por tener ecosistemas distintos, San Juan es costera e Iquitos está rodeada de selva amazónica)
#  Se unieron Train y Test al principio porque, para calcular el Lag o el promedio de las primeras semanas del Test,
#  se necesitaban los datos de las últimas semanas del Train. Si no, las primeras semanas del Test se habrían quedado en blanco.
train_full <- dataset[!is.na(total_cases)]
test_full <- dataset[is.na(total_cases)]

train_sj <- train_full[city == "sj"]
test_sj <- test_full[city == "sj"]
train_iq <- train_full[city == "iq"]
test_iq <- test_full[city == "iq"]

# LightGBM

In [11]:
# Preparar Matrices para LightGBM
features_lgb <- setdiff(names(train_sj), c("city", "year", "weekofyear", "total_cases"))

dtrain_sj <- lgb.Dataset(data = as.matrix(train_sj[, ..features_lgb]), label = train_sj$total_cases)
dtrain_iq <- lgb.Dataset(data = as.matrix(train_iq[, ..features_lgb]), label = train_iq$total_cases)

In [12]:
# Entrenamiento y optimización bayesiana para LightGBM

In [13]:
# Si ya se hizo antes toma el resultado con los mejores hiperparametros que se guardó en un .txt (para reducir tiempo de procesamiento más que  nada)

archivo_params <- "mejores_parametros_lgb.txt"

if (file.exists(archivo_params)) {
  cat("¡Archivo de parámetros encontrado! Cargando configuración...\n")

  # Lee archivo y extrae valores
  lineas <- readLines(archivo_params)
  parametros <- list(
    objective = "regression_l1",
    metric = "mae",
    verbosity = -1,
    bagging_freq = 5
  )

  for(linea in lineas) {
    partes <- strsplit(linea, "=")[[1]]
    if(length(partes) == 2) {
      parametros[[partes[1]]] <- as.numeric(partes[2])
    }
  }
  if(!is.null(parametros$num_leaves)) parametros$num_leaves <- as.integer(parametros$num_leaves)

} else {

#Si no está el archivo con los parñametros, inicia la optimización bayesiana....

  cat("No se encontró archivo de parámetros. Iniciando Optimización Bayesiana (este proceso tarda...)\n")


 #DEFINIR FUNCIÓN DE EVALUACIÓN
  # rBayesianOptimization siempre busca MAXIMIZAR. Como nuestro error (MAE) debe ser lo más bajo posible, busscamos que devuelta el MAE en negativo (-MAE).
  lgb_eval_fun <- function(learning_rate, num_leaves, feature_fraction, bagging_fraction) {

    pars <- list(
      objective = "regression_l1",
      metric = "mae",
      learning_rate = learning_rate,
      num_leaves = as.integer(num_leaves),
      feature_fraction = feature_fraction,
      bagging_fraction = bagging_fraction,
      bagging_freq = 5,
      verbosity = -1
    )

    # Usamos Validación Cruzada (CV) sobre San Juan para encontrar los mejores parámetros
    cv_modelo <- lgb.cv(
      params = pars,
      data = dtrain_sj,
      nrounds = 200,          # Límite de árboles
      nfold = 3,              # 3 particiones
      early_stopping_rounds = 20, # Frena si no mejora en 20 iteraciones
      showsd = FALSE
    )

    mejor_mae <- cv_modelo$best_score
    return(list(Score = -mejor_mae, Pred = 0))
  }

  #Definir los límites de busqueda (Bounds)
  limites <- list(
    learning_rate = c(0.01, 0.15),
    num_leaves = c(15L, 63L),  # L es porque es un rango de números enteros
    feature_fraction = c(0.5, 1.0),
    bagging_fraction = c(0.5, 1.0)
  )

  #EJECUCION DE LA OPTIMIZACION
  set.seed(42)
  opt_res <- BayesianOptimization(
    lgb_eval_fun,
    bounds = limites,
    init_points = 5,  # Puntos aleatorios iniciales
    n_iter = 15,      # numero de iteraciones de búsqueda
    acq = "ucb"
  )

  # Lista con los mejores parámetros
  mejores_pars <- opt_res$Best_Par
  parametros <- list(
    objective = "regression_l1",
    metric = "mae",
    learning_rate = mejores_pars["learning_rate"],
    num_leaves = as.integer(mejores_pars["num_leaves"]),
    feature_fraction = mejores_pars["feature_fraction"],
    bagging_fraction = mejores_pars["bagging_fraction"],
    bagging_freq = 5,
    verbosity = -1
  )

  # Guarda los resultados en un .txt para no volver a correrlo si ya se dispone de los resultados.
  cat("¡Optimización finalizada! Guardando los mejores parámetros en:", archivo_params, "\n")
  lineas_guardar <- c(
    paste0("learning_rate=", parametros$learning_rate),
    paste0("num_leaves=", parametros$num_leaves),
    paste0("feature_fraction=", parametros$feature_fraction),
    paste0("bagging_fraction=", parametros$bagging_fraction)
  )
  writeLines(lineas_guardar, archivo_params)
}

¡Archivo de parámetros encontrado! Cargando configuración...


In [14]:


# Entrenamiento con los parámetros
cat("\nEntrenando los modelos con los parámetros óptimos...\n")

# NOTA: nrounds un poco más alto porque LightGBM es robusto si los parámetros son buenos
modelo_sj <- lgb.train(params = parametros, data = dtrain_sj, nrounds = 300)
modelo_iq <- lgb.train(params = parametros, data = dtrain_iq, nrounds = 300)

cat("¡Modelos entrenados! Evaluando métrica interna MAE estimada en Train...\n")
pred_train_sj <- predict(modelo_sj, as.matrix(train_sj[, ..features_lgb]))
mae_sj <- mean(abs(train_sj$total_cases - pred_train_sj))
cat(">> MAE Interno San Juan estimado:", round(mae_sj, 2), "\n")

pred_train_iq <- predict(modelo_iq, as.matrix(train_iq[, ..features_lgb]))
mae_iq <- mean(abs(train_iq$total_cases - pred_train_iq))
cat(">> MAE Interno Iquitos estimado:", round(mae_iq, 2), "\n")


Entrenando los modelos con los parámetros óptimos...
¡Modelos entrenados! Evaluando métrica interna MAE estimada en Train...
>> MAE Interno San Juan estimado: 9.42 
>> MAE Interno Iquitos estimado: 2.31 


In [15]:
# Predicción Final
cat("Generando predicciones finales...\n")
pred_test_sj <- round(predict(modelo_sj, as.matrix(test_sj[, ..features_lgb])))
pred_test_iq <- round(predict(modelo_iq, as.matrix(test_iq[, ..features_lgb])))

test_sj$total_cases <- ifelse(pred_test_sj < 0, 0, pred_test_sj)
test_iq$total_cases <- ifelse(pred_test_iq < 0, 0, pred_test_iq)

sub_lgb <- rbindlist(list(test_sj[, .(city, year, weekofyear, total_cases)],
                          test_iq[, .(city, year, weekofyear, total_cases)]))

Generando predicciones finales...


In [16]:
# Exportación
molde <- fread("submission_format.csv")
molde[sub_lgb, total_cases := i.total_cases, on=.(city, year, weekofyear)]
molde[is.na(total_cases), total_cases := 0] # Blindaje contra NAs

fwrite(molde, "submission_lightgbm.csv")
cat("Archivo 'submission_lightgbm.csv' creado.\n")

Archivo 'submission_lightgbm.csv' creado.


#SVM

In [17]:
# MODELO 2: Support Vector Machine (SVM)

cat("\n--- Entrenando Modelo 2: SVM ---\n")

# Columnas para que sea independiente
features_model <- setdiff(names(train_sj), c("city", "year", "weekofyear", "total_cases"))




--- Entrenando Modelo 2: SVM ---


In [18]:
#optimización de parámetros

archivo_params_svm <- "mejores_parametros_svm.txt"

# Definimos las columnas predictoras
features_model <- setdiff(names(train_sj), c("city", "year", "weekofyear", "total_cases"))

if (file.exists(archivo_params_svm)) {
  cat("¡Archivo de parámetros SVM encontrado! Cargando configuración guardada...\n")

  # Carga parámetros desde el archivo
  lineas_svm <- readLines(archivo_params_svm)
  p_svm <- list()
  for(linea in lineas_svm) {
    partes <- strsplit(linea, "=")[[1]]
    if(length(partes) == 2) p_svm[[partes[1]]] <- as.numeric(partes[2])
  }

  # 2. Entrena modelo con parámetros guardados
  modelo_svm_sj <- svm(x = as.matrix(train_sj[, ..features_model]),
                       y = train_sj$total_cases,
                       type = "eps-regression", kernel = "radial", scale = TRUE,
                       cost = p_svm$cost_sj, gamma = p_svm$gamma_sj)

  modelo_svm_iq <- svm(x = as.matrix(train_iq[, ..features_model]),
                       y = train_iq$total_cases,
                       type = "eps-regression", kernel = "radial", scale = TRUE,
                       cost = p_svm$cost_iq, gamma = p_svm$gamma_iq)

  cat("Modelos San Juan e Iquitos entrenados con parámetros óptimos cargados.\n")

} else {
  cat("No se encontró archivo de parámetros SVM. Iniciando Grid Search (esto puede tardar)...\n")

  # 1. Optimización para San Juan
  cat("Optimizando San Juan...\n")
  tune_sj <- tune(svm, train.x = as.matrix(train_sj[, ..features_model]), train.y = train_sj$total_cases,
                  type = "eps-regression", kernel = "radial", scale = TRUE,
                  ranges = list(cost = c(0.1, 1, 10, 50), gamma = c(0.01, 0.05, 0.1)))
  modelo_svm_sj <- tune_sj$best.model

  # 2. Optimización para Iquitos
  cat("Optimizando Iquitos...\n")
  tune_iq <- tune(svm, train.x = as.matrix(train_iq[, ..features_model]), train.y = train_iq$total_cases,
                  type = "eps-regression", kernel = "radial", scale = TRUE,
                  ranges = list(cost = c(0.1, 1, 10, 50), gamma = c(0.01, 0.05, 0.1)))
  modelo_svm_iq <- tune_iq$best.model

  # 3. Guarda resultados en .txt
  cat("Guardando mejores parámetros en:", archivo_params_svm, "\n")
  lineas_guardar_svm <- c(
    paste0("cost_sj=", modelo_svm_sj$cost),
    paste0("gamma_sj=", modelo_svm_sj$gamma),
    paste0("cost_iq=", modelo_svm_iq$cost),
    paste0("gamma_iq=", modelo_svm_iq$gamma)
  )
  writeLines(lineas_guardar_svm, archivo_params_svm)
}


¡Archivo de parámetros SVM encontrado! Cargando configuración guardada...
Modelos San Juan e Iquitos entrenados con parámetros óptimos cargados.


In [19]:

# Evaluación interna MAE
pred_train_svm_sj <- predict(modelo_svm_sj, as.matrix(train_sj[, ..features_model]))
pred_train_svm_iq <- predict(modelo_svm_iq, as.matrix(train_iq[, ..features_model]))

cat("\n>> MAE Interno SVM San Juan:", round(mean(abs(train_sj$total_cases - pred_train_svm_sj)), 2), "\n")
cat(">> MAE Interno SVM Iquitos:", round(mean(abs(train_iq$total_cases - pred_train_svm_iq)), 2), "\n")

# Predicción sobre el Test Set con redondeo y control de negativos
pred_test_svm_sj <- round(predict(modelo_svm_sj, as.matrix(test_sj[, ..features_model])))
pred_test_svm_sj <- ifelse(pred_test_svm_sj < 0, 0, pred_test_svm_sj)

pred_test_svm_iq <- round(predict(modelo_svm_iq, as.matrix(test_iq[, ..features_model])))
pred_test_svm_iq <- ifelse(pred_test_svm_iq < 0, 0, pred_test_svm_iq)

cat("SVM completado!\n")


>> MAE Interno SVM San Juan: 4.84 
>> MAE Interno SVM Iquitos: 2.74 
SVM completado!


In [20]:
# Exportación de resultados
cat("\nGenerando archivo de Submission SVM...\n")

sub_svm <- rbindlist(list(
  test_sj[, .(city, year, weekofyear, total_cases = pred_test_svm_sj)],
  test_iq[, .(city, year, weekofyear, total_cases = pred_test_svm_iq)]
))

molde <- fread("submission_format.csv")
molde[sub_svm, total_cases := i.total_cases, on=.(city, year, weekofyear)]
molde[is.na(total_cases), total_cases := 0]

fwrite(molde, "submission_svm.csv")
cat("Archivo 'submission_svm.csv' creado.\n")


Generando archivo de Submission SVM...
Archivo 'submission_svm.csv' creado.


# Redes Neuronales

In [21]:
# MODELO 3 Redes Neuronales Artificiales - ANN
# -------------------------------------------------------

cat("\n--- Entrenando Modelo 3: Red Neuronal ---\n")

# Armé copias independientes para trabajar con tranquilidad
train_sj_nn <- copy(train_sj)
test_sj_nn <- copy(test_sj)
train_iq_nn <- copy(train_iq)
test_iq_nn <- copy(test_iq)

# Normalización de variables predictoras
normalize <- function(x) {
  min_val <- min(x, na.rm = TRUE)
  max_val <- max(x, na.rm = TRUE)
  if(max_val - min_val == 0) return(rep(0, length(x)))
  return ((x - min_val) / (max_val - min_val))
}

cols_todas <- setdiff(names(train_sj_nn), c("city", "year", "weekofyear", "week_start_date", "total_cases"))

train_sj_nn[, (cols_todas) := lapply(.SD, normalize), .SDcols=cols_todas]
test_sj_nn[, (cols_todas) := lapply(.SD, normalize), .SDcols=cols_todas]
train_iq_nn[, (cols_todas) := lapply(.SD, normalize), .SDcols=cols_todas]
test_iq_nn[, (cols_todas) := lapply(.SD, normalize), .SDcols=cols_todas]

# Escalado de la variable objetivo (Quedan guardados los máximos y los mínimos para des-escalar al finalizar para el submit)
min_cases_sj <- min(train_sj_nn$total_cases, na.rm=TRUE)
max_cases_sj <- max(train_sj_nn$total_cases, na.rm=TRUE)
train_sj_nn[, total_cases := (total_cases - min_cases_sj) / (max_cases_sj - min_cases_sj)]

min_cases_iq <- min(train_iq_nn$total_cases, na.rm=TRUE)
max_cases_iq <- max(train_iq_nn$total_cases, na.rm=TRUE)
train_iq_nn[, total_cases := (total_cases - min_cases_iq) / (max_cases_iq - min_cases_iq)]

# Definición de las features excluyendo las variables de control y la objetivo
features_model <- setdiff(names(train_sj_nn), c("city", "year", "weekofyear", "total_cases", "week_start_date"))
num_inputs <- length(features_model)

x_train_sj <- as.matrix(train_sj_nn[, ..features_model])
y_train_sj <- as.numeric(train_sj_nn$total_cases)
x_train_iq <- as.matrix(train_iq_nn[, ..features_model])
y_train_iq <- as.numeric(train_iq_nn$total_cases)
x_test_sj <- as.matrix(test_sj_nn[, ..features_model])
x_test_iq <- as.matrix(test_iq_nn[, ..features_model])


--- Entrenando Modelo 3: Red Neuronal ---


In [28]:
# Torch requiere que todo sea numérico y en formato de tensor (matrices)
x_sj_t <- torch_tensor(x_train_sj, dtype = torch_float32())
y_sj_t <- torch_tensor(matrix(y_train_sj, ncol = 1), dtype = torch_float32())

x_iq_t <- torch_tensor(x_train_iq, dtype = torch_float32())
y_iq_t <- torch_tensor(matrix(y_train_iq, ncol = 1), dtype = torch_float32())

x_test_sj_t <- torch_tensor(x_test_sj, dtype = torch_float32())
x_test_iq_t <- torch_tensor(x_test_iq, dtype = torch_float32())

# Se define la arquitectura de la red (este paso es equivalente al Sequential() de Keras3 que es la function que falla en R)
dengue_mlp <- nn_module(
  "DengueMLP",
  initialize = function(num_inputs, nodes1, nodes2, drop_rate) {
    self$fc1 <- nn_linear(num_inputs, nodes1)
    self$drop1 <- nn_dropout(drop_rate)
    self$fc2 <- nn_linear(nodes1, nodes2)
    self$drop2 <- nn_dropout(drop_rate)
    self$out <- nn_linear(nodes2, 1)
  },
  forward = function(x) {
    x %>%
      self$fc1() %>% nnf_relu() %>% self$drop1() %>%
      self$fc2() %>% nnf_relu() %>% self$drop2() %>%
      self$out()
  }
)

# Búsqueda o carga de parámetros a partir de un .txt
archivo_params_torch <- "mejores_parametros_torch.txt"

if (file.exists(archivo_params_torch)) {
  cat("¡Archivo de parámetros Torch encontrado! Cargando configuración guardada...\n")

  lineas_t <- readLines(archivo_params_torch)
  p_t <- list()
  for(linea in lineas_t) {
    partes <- strsplit(linea, "=")[[1]]
    if(length(partes) == 2) p_t[[partes[1]]] <- as.numeric(partes[2])
  }

} else {
  cat("No se encontró archivo. Iniciando Optimización Bayesiana para Torch...\n")

  # Función de evaluación para la Optimización Bayesiana
  torch_eval_fun <- function(nodes1, nodes2, dropout, lr) {

    # Seed para reproducibilidad.... 42! Respuesta a la pregunta fundamental sobre la vida, el universo y todo lo demás
    torch_manual_seed(42)

    # Se configura el modelo y se entrena usando el package 'luz'
    fitted <- dengue_mlp %>%
      setup(
        loss = nn_l1_loss(), # L1 Loss es el MAE matemático
        optimizer = optim_adam
      ) %>%
      set_hparams(num_inputs = num_inputs, nodes1 = as.integer(nodes1),
                  nodes2 = as.integer(nodes2), drop_rate = dropout) %>%
      set_opt_hparams(lr = lr) %>%
      fit(
        data = list(x_sj_t, y_sj_t),
        epochs = 50,
        valid_data = 0.2,
        callbacks = list(luz_callback_early_stopping(monitor = "valid_loss", patience = 10)),
        verbose = FALSE
      )

    metrics_df <- get_metrics(fitted)
    df_valid <- metrics_df[metrics_df$set == "valid", ]

    if ("value" %in% colnames(df_valid)) {
      mejor_val_mae <- min(df_valid$value, na.rm = TRUE)
    } else {
      es_numero <- sapply(df_valid, is.numeric)
      es_numero[names(es_numero) == "epoch"] <- FALSE
      col_num <- names(df_valid)[es_numero][1]
      mejor_val_mae <- min(df_valid[[col_num]], na.rm = TRUE)
    }

    mejor_val_mae <- as.numeric(mejor_val_mae)

    return(list(Score = -mejor_val_mae, Pred = 0))
  }

  # Límites de búsqueda
  limites_torch <- list(
    nodes1 = c(32L, 256L),
    nodes2 = c(16L, 128L),
    dropout = c(0.1, 0.5),
    lr = c(0.0001, 0.01)
  )

  # Optimización bayesiana
  set.seed(42)
  opt_res_torch <- BayesianOptimization(
    torch_eval_fun,
    bounds = limites_torch,
    init_points = 5,
    n_iter = 10,
    acq = "ucb"
  )

  # Se guardan los mejores
  mejores_t <- opt_res_torch$Best_Par
  p_t <- list(
    nodes1 = as.integer(mejores_t["nodes1"]),
    nodes2 = as.integer(mejores_t["nodes2"]),
    dropout = mejores_t["dropout"],
    lr = mejores_t["lr"]
  )

  cat("\n¡Guardando los mejores parámetros Torch en:", archivo_params_torch, "\n")
  lineas_guardar_t <- c(
    paste0("nodes1=", p_t$nodes1),
    paste0("nodes2=", p_t$nodes2),
    paste0("dropout=", p_t$dropout),
    paste0("lr=", p_t$lr)
  )
  writeLines(lineas_guardar_t, archivo_params_torch)
}

# Entrenanamiento de la red


cat("\nEntrenando Red Neuronal Nativa (Modelo Definitivo)...\n")

crear_y_entrenar_torch <- function(x_tensor, y_tensor) {
  torch_manual_seed(42)
  modelo_final <- dengue_mlp %>%
    setup(loss = nn_l1_loss(), optimizer = optim_adam) %>%
    set_hparams(num_inputs = num_inputs, nodes1 = p_t$nodes1,
                nodes2 = p_t$nodes2, drop_rate = p_t$dropout) %>%
    set_opt_hparams(lr = p_t$lr) %>%
    fit(
      data = list(x_tensor, y_tensor),
      epochs = 150,
      verbose = FALSE
    )
  return(modelo_final)
}

model_sj_torch <- crear_y_entrenar_torch(x_sj_t, y_sj_t)
model_iq_torch <- crear_y_entrenar_torch(x_iq_t, y_iq_t)

No se encontró archivo. Iniciando Optimización Bayesiana para Torch...
elapsed = 8.687	Round = 1	nodes1 = 237.0000	nodes2 = 74.0000	dropout = 0.2830967	lr = 0.009406144	Value = -0.04364479 
elapsed = 11.21	Round = 2	nodes1 = 242.0000	nodes2 = 98.0000	dropout = 0.3876449	lr = 0.009784442	Value = -0.04543082 
elapsed = 10.105	Round = 3	nodes1 = 96.0000	nodes2 = 31.0000	dropout = 0.4738689	lr = 0.001263125	Value = -0.04952896 
elapsed = 8.341	Round = 4	nodes1 = 218.0000	nodes2 = 90.0000	dropout = 0.2021715	lr = 0.004802471	Value = -0.04666611 
elapsed = 6.831	Round = 5	nodes1 = 176.0000	nodes2 = 95.0000	dropout = 0.2849171	lr = 0.005647294	Value = -0.04773641 
elapsed = 3.903	Round = 6	nodes1 = 256.0000	nodes2 = 36.0000	dropout = 0.4478392	lr = 0.0100	Value = -0.05665614 
elapsed = 7.601	Round = 7	nodes1 = 256.0000	nodes2 = 128.0000	dropout = 0.1000	lr = 0.0100	Value = -0.04110418 
elapsed = 5.617	Round = 8	nodes1 = 32.0000	nodes2 = 128.0000	dropout = 0.3576296	lr = 0.0100	Value = -0.0571

In [29]:
# Evaluación y des-escalado
# Convertimos las predicciones del tensor a vector numérico estándar de R
pred_train_sj <- as.numeric(predict(model_sj_torch, x_sj_t))
pred_train_sj_real <- (pred_train_sj * (max_cases_sj - min_cases_sj)) + min_cases_sj

pred_train_iq <- as.numeric(predict(model_iq_torch, x_iq_t))
pred_train_iq_real <- (pred_train_iq * (max_cases_iq - min_cases_iq)) + min_cases_iq

cat("\n>> MAE Interno Torch San Juan:", round(mean(abs(train_sj$total_cases - pred_train_sj_real)), 2), "\n")
cat(">> MAE Interno Torch Iquitos:", round(mean(abs(train_iq$total_cases - pred_train_iq_real)), 2), "\n")


>> MAE Interno Torch San Juan: 24.72 
>> MAE Interno Torch Iquitos: 4.99 


In [30]:
# Predicción
cat("\nGenerando predicciones sobre el Test Set...\n")
pred_test_sj <- round((as.numeric(predict(model_sj_torch, x_test_sj_t)) * (max_cases_sj - min_cases_sj)) + min_cases_sj)
pred_test_iq <- round((as.numeric(predict(model_iq_torch, x_test_iq_t)) * (max_cases_iq - min_cases_iq)) + min_cases_iq)

# Control de negativos
pred_test_sj <- ifelse(pred_test_sj < 0, 0, pred_test_sj)
pred_test_iq <- ifelse(pred_test_iq < 0, 0, pred_test_iq)


Generando predicciones sobre el Test Set...


In [31]:
#Exportación
sub_nn <- rbindlist(list(
  test_sj_nn[, .(city, year, weekofyear, total_cases = pred_test_sj)],
  test_iq_nn[, .(city, year, weekofyear, total_cases = pred_test_iq)]
))

molde <- fread("submission_format.csv")
molde[sub_nn, total_cases := i.total_cases, on=.(city, year, weekofyear)]
molde[is.na(total_cases), total_cases := 0]

fwrite(molde, "submission_nn.csv")
cat("Archivo 'submission_nn.csv' creado con éxito.\n")

Archivo 'submission_nn.csv' creado con éxito.


# Submission a DrivenData

Verificación de la integridad de los archivos para DrivenData

In [32]:
cat("Revision del formato para archivos para DrivenData...\n\n")

# Formato de submisision
molde <- fread("submission_format.csv")
filas_esperadas <- nrow(molde)
cols_esperadas <- c("city", "year", "weekofyear", "total_cases")

revisar_archivo <- function(archivo_in) {
  if(file.exists(archivo_in)) {
    df <- fread(archivo_in)
    alertas <- c()

    # Cantidad de columnas
    faltantes <- setdiff(cols_esperadas, names(df))
    sobrantes <- setdiff(names(df), cols_esperadas)

    if(length(faltantes) > 0) {
      alertas <- c(alertas, paste("Faltan columnas:", paste(faltantes, collapse=", ")))
    }
    if(length(sobrantes) > 0) {
      alertas <- c(alertas, paste("Sobran columnas:", paste(sobrantes, collapse=", ")))
    }

    # Cantidad de filas
    if(nrow(df) != filas_esperadas) {
      alertas <- c(alertas, paste("Hay", nrow(df), "files pero deberían ser", filas_esperadas))
    }

    # Valores NA en las predicciones
    if("total_cases" %in% names(df) && any(is.na(df$total_cases))) {
      cantidad_na <- sum(is.na(df$total_cases))
      alertas <- c(alertas, paste("Hay", cantidad_na, "valores NA en total_cases."))
    }

    # Mostrar resultados del archivo
    if(length(alertas) > 0) {
      cat("Error en el archivo:", archivo_in, "\n")
      for(a in alertas) cat("   ->", a, "\n")
      cat("\n")
    } else {
      cat("Formato correcto:", archivo_in, "- Listo para subir.\n\n")
    }

  } else {
    cat("Archivo no encontrado:", archivo_in, "\n\n")
  }
}

# Revisión para los 3 modelos generados
cat("--- Resultados de la Validación ---\n")
revisar_archivo("submission_lightgbm.csv")
revisar_archivo("submission_svm.csv")
revisar_archivo("submission_nn.csv")
cat("-----------------------------------\n")

Revision del formato para archivos para DrivenData...

--- Resultados de la Validación ---
Formato correcto: submission_lightgbm.csv - Listo para subir.

Formato correcto: submission_svm.csv - Listo para subir.

Formato correcto: submission_nn.csv - Listo para subir.

-----------------------------------


In [ ]:
# Por el momento el submit es a mano, a futuro podría sumarse un envío automatizado vía JSON